In [ ]:
"""# 01_demographics_and_cohort_data.ipynb

**Purpose:** This notebook calculates summary statistics and extracts demographic data (Age, Biological Sex, and Mortality Outcomes) across the seven harmonized GEO cohorts. 
             It maps cohort-specific metadata columns to a unified schema to generate the summary statistics required for Table 1 of the manuscript.

**Outputs:** - Cohort patient distributions.
- Mortality cross-tabulations.
- Pooled and stratified demographic statistics (Median/IQR for Age, Percentages for Sex)."""

In [3]:
import pandas as pd
from pathlib import Path

# ==========================================
# 1. COHORT DISTRIBUTION ANALYSIS
# ==========================================
# Resolve paths assuming execution from the /notebooks/ directory
BASE_DIR = Path.cwd().parent
LABELS_PATH = BASE_DIR / "data" / "processed" / "matrices" / "master_clinical_labels.csv"

if not LABELS_PATH.exists():
    raise FileNotFoundError(f"[ERROR] Clinical labels not found at: {LABELS_PATH}")

df_labels = pd.read_csv(LABELS_PATH)

# Aggregate patient counts per cohort
cohort_counts = df_labels['Dataset'].value_counts().reset_index()
cohort_counts.columns = ['Cohort', 'Patient_Count']

print("==========================================")
print("COHORT PATIENT DISTRIBUTION")
print("==========================================")
print(f"Total Validated Patients: {len(df_labels)}\n")
print(cohort_counts.to_string(index=False))

COHORT PATIENT DISTRIBUTION
Total Validated Patients: 1636

   Cohort  Patient_Count
 GSE65682            479
GSE185263            345
GSE236713            324
GSE272769            161
 GSE54514            127
 GSE95233            102
 GSE26440             98


In [4]:
# ==========================================
# 2. MORTALITY BREAKDOWN PER COHORT
# ==========================================
# Map binary labels to clinical outcomes
df_labels['Outcome'] = df_labels['Mortality'].map({0: 'Survivor', 1: 'Non-Survivor'})

# Generate cross-tabulation matrix
breakdown_matrix = pd.crosstab(
    index=df_labels['Dataset'], 
    columns=df_labels['Outcome'], 
    margins=True, 
    margins_name="Total"
)

print("==========================================")
print("MORTALITY DISTRIBUTION MATRIX")
print("==========================================")
print(breakdown_matrix)

MORTALITY DISTRIBUTION MATRIX
Outcome    Non-Survivor  Survivor  Total
Dataset                                 
GSE185263            52       293    345
GSE236713            59       265    324
GSE26440             17        81     98
GSE272769            60       101    161
GSE54514             31        96    127
GSE65682            114       365    479
GSE95233             34        68    102
Total               367      1269   1636


In [5]:
import numpy as np
import re

# ==========================================
# 3. COHORT-LEVEL DEMOGRAPHIC EXTRACTION
# ==========================================
META_DIR = BASE_DIR / "data" / "raw" / "geo_metadata"

# Define mappings for heterogeneous metadata columns across datasets
COHORT_MAPPING = {
    'GSE185263': {'age': 'age', 'sex': 'sex'},
    'GSE236713': {'age': None, 'sex': 'gender'},      # Age not reported
    'GSE26440':  {'age': 'age (years)', 'sex': None}, # Sex not reported
    'GSE272769': {'age': 'age', 'sex': 'sex'},
    'GSE54514':  {'age': 'age (years)', 'sex': 'gender'},
    'GSE65682':  {'age': 'age', 'sex': 'gender'},
    'GSE95233':  {'age': 'age', 'sex': 'gender'}
}

def extract_age(val) -> float:
    """Parses numerical age from heterogenous string formats."""
    if pd.isna(val): 
        return np.nan
    match = re.search(r'\d+(\.\d+)?', str(val))
    return float(match.group()) if match else np.nan

def clean_sex(val) -> str:
    """Standardizes biological sex annotations."""
    if pd.isna(val): 
        return "Unknown"
    v = str(val).lower().strip()
    if v in ['m', 'male']: 
        return 'Male'
    if v in ['f', 'female']: 
        return 'Female'
    return "Unknown"

print("==========================================")
print("DEMOGRAPHICS BY COHORT (FOR SUPPLEMENTARY)")
print("==========================================\n")

for cohort, mapping in COHORT_MAPPING.items():
    # Isolate valid patients for the current cohort
    valid_patients = df_labels[df_labels['Dataset'] == cohort]['Patient_ID'].tolist()
    
    # Load and filter raw metadata
    meta_path = META_DIR / f"{cohort}_metadata.csv"
    df_meta = pd.read_csv(meta_path)
    df_meta = df_meta[df_meta['Patient_ID'].isin(valid_patients)]
    
    # Process Age
    if mapping['age'] and mapping['age'] in df_meta.columns:
        ages = df_meta[mapping['age']].apply(extract_age).dropna()
        if len(ages) > 0:
            age_str = f"{ages.median():.1f} [{ages.quantile(0.25):.1f}-{ages.quantile(0.75):.1f}]"
        else:
            age_str = "Values Unparseable"
    else:
        age_str = "Not Reported"
        
    # Process Sex
    if mapping['sex'] and mapping['sex'] in df_meta.columns:
        sexes = df_meta[mapping['sex']].apply(clean_sex)
        male_c = (sexes == 'Male').sum()
        female_c = (sexes == 'Female').sum()
        sex_str = f"Male: {male_c}, Female: {female_c}"
    else:
        sex_str = "Not Reported"
        
    print(f"[{cohort}] (N = {len(valid_patients)})")
    print(f"  Age (Median [IQR]) : {age_str}")
    print(f"  Sex (Distribution) : {sex_str}\n")

DEMOGRAPHICS BY COHORT (FOR SUPPLEMENTARY)

[GSE185263] (N = 345)
  Age (Median [IQR]) : 61.0 [44.0-73.0]
  Sex (Distribution) : Male: 201, Female: 144

[GSE236713] (N = 324)
  Age (Median [IQR]) : Not Reported
  Sex (Distribution) : Male: 167, Female: 157

[GSE26440] (N = 98)
  Age (Median [IQR]) : 2.2 [1.0-5.9]
  Sex (Distribution) : Not Reported

[GSE272769] (N = 161)
  Age (Median [IQR]) : 63.0 [54.0-73.0]
  Sex (Distribution) : Male: 74, Female: 87

[GSE54514] (N = 127)
  Age (Median [IQR]) : 60.0 [51.0-71.0]
  Sex (Distribution) : Male: 52, Female: 75

[GSE65682] (N = 479)
  Age (Median [IQR]) : 63.0 [53.0-71.0]
  Sex (Distribution) : Male: 272, Female: 207

[GSE95233] (N = 102)
  Age (Median [IQR]) : 66.0 [53.2-74.2]
  Sex (Distribution) : Male: 62, Female: 36



In [6]:
# ==========================================
# 4. POOLED DEMOGRAPHIC STATISTICS (TABLE 1)
# ==========================================
pooled_data = []

for cohort, mapping in COHORT_MAPPING.items():
    # Isolate outcome labels for the cohort
    valid_patients = df_labels[df_labels['Dataset'] == cohort][['Patient_ID', 'Outcome']].copy()
    
    # Load metadata
    meta_path = META_DIR / f"{cohort}_metadata.csv"
    df_meta = pd.read_csv(meta_path)
    
    # Merge outcomes with metadata
    df_merged = pd.merge(valid_patients, df_meta, on='Patient_ID', how='inner')
    
    # Apply standardizations
    if mapping['age'] and mapping['age'] in df_merged.columns:
        df_merged['Clean_Age'] = df_merged[mapping['age']].apply(extract_age)
    else:
        df_merged['Clean_Age'] = np.nan
        
    if mapping['sex'] and mapping['sex'] in df_merged.columns:
        df_merged['Clean_Sex'] = df_merged[mapping['sex']].apply(clean_sex)
    else:
        df_merged['Clean_Sex'] = "Unknown"
        
    pooled_data.append(df_merged[['Patient_ID', 'Outcome', 'Clean_Age', 'Clean_Sex']])

master_pool = pd.concat(pooled_data, ignore_index=True)

def print_summary_stats(df_subset: pd.DataFrame, group_name: str):
    """Calculates and formats demographic summary statistics for a given cohort subset."""
    n = len(df_subset)
    
    # Calculate Age Statistics
    ages = df_subset['Clean_Age'].dropna()
    if len(ages) > 0:
        age_str = f"{ages.median():.1f} [{ages.quantile(0.25):.1f}-{ages.quantile(0.75):.1f}]"
    else:
        age_str = "N/A"
    age_missing = df_subset['Clean_Age'].isna().sum()
    
    # Calculate Sex Statistics
    males = (df_subset['Clean_Sex'] == 'Male').sum()
    females = (df_subset['Clean_Sex'] == 'Female').sum()
    unknowns = (df_subset['Clean_Sex'] == 'Unknown').sum()
    
    print(f"\n{group_name} (N = {n})")
    print("-" * 45)
    print(f"Age (Median [IQR])     : {age_str}  *(Missing Data: {age_missing})*")
    print(f"Sex - Male (%)         : {males} ({(males/n)*100:.1f}%)")
    print(f"Sex - Female (%)       : {females} ({(females/n)*100:.1f}%)")
    print(f"Sex - Unreported       : {unknowns}")

print("==========================================")
print("POOLED COHORT DEMOGRAPHICS (TABLE 1)")
print("==========================================")
print_summary_stats(master_pool, "TOTAL COHORT")
print_summary_stats(master_pool[master_pool['Outcome'] == 'Survivor'], "SURVIVORS")
print_summary_stats(master_pool[master_pool['Outcome'] == 'Non-Survivor'], "NON-SURVIVORS")

POOLED COHORT DEMOGRAPHICS (TABLE 1)

TOTAL COHORT (N = 1636)
---------------------------------------------
Age (Median [IQR])     : 61.0 [47.0-71.0]  *(Missing Data: 324)*
Sex - Male (%)         : 828 (50.6%)
Sex - Female (%)       : 706 (43.2%)
Sex - Unreported       : 102

SURVIVORS (N = 1269)
---------------------------------------------
Age (Median [IQR])     : 60.0 [42.8-70.0]  *(Missing Data: 265)*
Sex - Male (%)         : 649 (51.1%)
Sex - Female (%)       : 539 (42.5%)
Sex - Unreported       : 81

NON-SURVIVORS (N = 367)
---------------------------------------------
Age (Median [IQR])     : 65.0 [53.0-74.0]  *(Missing Data: 59)*
Sex - Male (%)         : 179 (48.8%)
Sex - Female (%)       : 167 (45.5%)
Sex - Unreported       : 21
